# Fine-tune Qwen3-4B-Thinking on Meddies Consultant RandomQA (Kaggle 2×T4)

This notebook fine-tunes **Qwen3-4B-Thinking** using [Unsloth](https://unsloth.ai/) on the `Meddies/meddies-consultant` → `RandomQA` subset.

- **Environment:** Kaggle with 2× NVIDIA T4 GPUs
- **Strategy:** DDP (Distributed Data Parallel) via `torchrun` for multi-GPU training
- **Dataset columns:** `question` (Vietnamese clinical question), `answer` (includes `<think>` reasoning)

> **How to run on Kaggle:**
> 1. Upload this notebook to Kaggle
> 2. Set Accelerator to **GPU T4 ×2**
> 3. Enable **Internet**
> 4. Run all cells

## 1. Installation

In [ ]:
%%capture
import os, re

# Kaggle / Colab / Local detection
if "KAGGLE_" in "".join(os.environ.keys()):
    # Kaggle environment
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, '0.0.34')
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
elif "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Local setup
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, '0.0.34')
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"

!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

## 2. Verify GPU Setup

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Number of GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} — {round(props.total_memory / 1024**3, 1)} GB")

assert torch.cuda.device_count() >= 2, "This notebook requires 2 GPUs (Kaggle T4 ×2)!"

## 3. Write the DDP Training Script

For multi-GPU DDP training on Kaggle (or any multi-GPU environment), we write a standalone `.py` script and launch it with `torchrun`. This is the recommended approach from [Unsloth DDP docs](https://unsloth.ai/docs/basics/multi-gpu-training-with-unsloth/ddp).

> **Key point:** Unsloth auto-enables DDP when training with >1 GPU via `torchrun`.

In [ ]:
%%writefile train_ddp.py
#!/usr/bin/env python3
"""
DDP Training Script for Qwen3-4B-Thinking on Meddies/meddies-consultant (RandomQA)
Launch with: torchrun --nproc_per_node=2 train_ddp.py
"""
import os
import torch
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

# ============================================================
# 1. HYPERPARAMETERS
# ============================================================
MODEL_NAME      = "unsloth/Qwen3-4B-Thinking-2507"  # Unsloth optimized version
MAX_SEQ_LENGTH  = 2048
LOAD_IN_4BIT    = True       # QLoRA: 4-bit quantization to fit in T4 VRAM

# LoRA config (Tối ưu cho việc giữ lại khả năng tư duy logic)
LORA_R          = 16
LORA_ALPHA      = 16         # Giảm xuống để tránh overfitting/bùng nổ gradient
LORA_DROPOUT    = 0
TARGET_MODULES  = ["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"]

# Training config
BATCH_SIZE      = 8          # per device
GRAD_ACCUM      = 1          # Effective batch size = 16
WARMUP_STEPS    = 150        # Tăng warmup cho đường cong Cosine mượt hơn
NUM_EPOCHS      = 1          # full epoch
MAX_STEPS       = -1         # -1 = use num_epochs; set to e.g. 100 for quick test
LEARNING_RATE   = 5e-5       # Giảm xuống mức an toàn cho tập dữ liệu lớn
LR_SCHEDULER    = "cosine"   # cosine annealing
WEIGHT_DECAY    = 0.05       # Tăng nhẹ để tránh overfit với 68k rows
SEED            = 3407

# Output & Tiết kiệm dung lượng Kaggle
OUTPUT_DIR      = "./outputs"
SAVE_STEPS      = 500        # Tránh lưu quá nhiều checkpoint gây tràn đĩa
LOGGING_STEPS   = 5          # Tăng nhẹ để log đỡ bị spam liên tục

# ============================================================
# 2. LOAD MODEL + TOKENIZER
# ============================================================
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit   = LOAD_IN_4BIT,
    load_in_8bit   = False,
    full_finetuning = False,
)

# ============================================================
# 3. ADD LoRA ADAPTERS
# ============================================================
model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_R,
    target_modules             = TARGET_MODULES,
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = LORA_DROPOUT,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",  # 30% less VRAM
    random_state               = SEED,
    use_rslora                 = False,
    loftq_config               = None,
)

# ============================================================
# 4. SETUP CHAT TEMPLATE (Qwen3-Thinking)
# ============================================================
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen3-thinking",
)

# ============================================================
# 5. LOAD & PREPARE DATASET
# ============================================================
dataset = load_dataset("Meddies/meddies-consultant", "RandomQA", split="train")
print(f"[Rank {os.environ.get('LOCAL_RANK', '0')}] Dataset loaded: {len(dataset)} examples")

def generate_conversation(examples):
    """Convert question/answer columns to Qwen3-Thinking chat format.
    
    The 'answer' column already contains <think>...</think> reasoning blocks,
    which aligns perfectly with the qwen3-thinking chat template.
    """
    questions = examples["question"]
    answers   = examples["answer"]
    conversations = []
    for question, answer in zip(questions, answers):
        conversations.append([
            {"role": "user",      "content": question},
            {"role": "assistant", "content": answer},
        ])
    return {"conversations": conversations}

dataset = dataset.map(generate_conversation, batched=True)

def formatting_prompts_func(examples):
    """Apply the Qwen3-Thinking chat template to conversations."""
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        )
        for convo in convos
    ]
    return {"text": texts}

import os
MAX_NUM_PROC = os.cpu_count()

dataset = dataset.map(
    formatting_prompts_func,
    batched = True,
    num_proc = MAX_NUM_PROC,
)

# ============================================================
# 6. CREATE TRAINER
# ============================================================
trainer = SFTTrainer(
    model     = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset  = None,
    args = SFTConfig(
        dataset_text_field          = "text",
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_steps                = WARMUP_STEPS,
        num_train_epochs            = NUM_EPOCHS,
        max_steps                   = MAX_STEPS,
        learning_rate               = LEARNING_RATE,
        lr_scheduler_type           = LR_SCHEDULER,
        weight_decay                = WEIGHT_DECAY,
        logging_steps               = LOGGING_STEPS,
        save_steps                  = SAVE_STEPS,
        save_total_limit            = 3,
        optim                       = "adamw_8bit",
        seed                        = SEED,
        output_dir                  = OUTPUT_DIR,
        report_to                   = "none",
        fp16         = not torch.cuda.is_bf16_supported(),
        bf16         = torch.cuda.is_bf16_supported(),
        # DDP settings (auto-configured by torchrun)
        ddp_find_unused_parameters  = False,
    ),
)

# ============================================================
# 7. TRAIN ON RESPONSES ONLY (mask user input from loss)
# ============================================================
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

# ============================================================
# 8. TRAIN!
# ============================================================
print("="*60)
print("Starting training...")
print(f"  Model:          {MODEL_NAME}")
print(f"  Dataset:        Meddies/meddies-consultant (RandomQA)")
print(f"  Num examples:   {len(dataset)}")
print(f"  LoRA:           r={LORA_R}, alpha={LORA_ALPHA} (scaling={LORA_ALPHA/LORA_R})")
print(f"  Batch/device:   {BATCH_SIZE}")
print(f"  Grad accum:     {GRAD_ACCUM}")
print(f"  Num GPUs:       {torch.cuda.device_count()}")
print(f"  Effective batch: {BATCH_SIZE * GRAD_ACCUM * torch.cuda.device_count()}")
print(f"  Learning rate:  {LEARNING_RATE}")
print(f"  Weight decay:   {WEIGHT_DECAY}")
print(f"  Warmup steps:   {WARMUP_STEPS}")
print("="*60)

# Resume training from an existing checkpoint
RESUME_CHECKPOINT = "/kaggle/input/models/tu4nhoang/ckpt-7249/pytorch/default/1/ckpt/qwen3-4b-medical-qa-lora/checkpoint-7000"

trainer_stats = trainer.train(
    resume_from_checkpoint=RESUME_CHECKPOINT
)

# ============================================================
# 9. SAVE MODEL (only on main process)
# ============================================================
local_rank = int(os.environ.get("LOCAL_RANK", 0))
if local_rank == 0:
    print("\n" + "="*60)
    print("Training complete! Saving model...")
    print("="*60)

    # Save LoRA adapters
    model.save_pretrained("qwen3_4b_thinking_meddies_lora")
    tokenizer.save_pretrained("qwen3_4b_thinking_meddies_lora")
    print("LoRA adapters saved to: qwen3_4b_thinking_meddies_lora/")

    # Save merged 16-bit model
    model.save_pretrained_merged(
        "qwen3_4b_thinking_meddies_merged",
        tokenizer,
        save_method="merged_16bit",
    )
    print("Merged 16-bit model saved to: qwen3_4b_thinking_meddies_merged/")

    # Save GGUF for deployment
    model.save_pretrained_gguf(
        "qwen3_4b_thinking_meddies_gguf",
        tokenizer,
        quantization_method="q4_k_m",
    )
    print("GGUF (q4_k_m) saved to: qwen3_4b_thinking_meddies_gguf/")

    print("\nAll done! 🎉")

## 4. Launch DDP Training with `torchrun`

We use `torchrun --nproc_per_node=2` to launch the training script across both T4 GPUs.

This will:
- Spawn 2 training processes (one per GPU)
- Each process gets its own copy of the model
- Data is split across GPUs automatically
- Gradients are synchronized via NCCL

In [ ]:
!torchrun --nproc_per_node=2 train_ddp.py

## 5. Verify Saved Models

In [ ]:
import os

for save_dir in [
    "qwen3_4b_thinking_meddies_lora",
    "qwen3_4b_thinking_meddies_merged",
    "qwen3_4b_thinking_meddies_gguf",
]:
    if os.path.exists(save_dir):
        files = os.listdir(save_dir)
        total_size = sum(
            os.path.getsize(os.path.join(save_dir, f))
            for f in files
            if os.path.isfile(os.path.join(save_dir, f))
        )
        print(f"📁 {save_dir}/")
        print(f"   Files: {len(files)} | Total size: {total_size / 1024**3:.2f} GB")
        for f in sorted(files)[:10]:
            fpath = os.path.join(save_dir, f)
            if os.path.isfile(fpath):
                print(f"   - {f} ({os.path.getsize(fpath) / 1024**2:.1f} MB)")
        if len(files) > 10:
            print(f"   ... and {len(files) - 10} more files")
        print()
    else:
        print(f"⚠️  {save_dir}/ not found")

## 6. Inference — Test the Fine-tuned Model

Load the LoRA adapter and run inference on a sample Vietnamese clinical question.

In [ ]:
from unsloth import FastLanguageModel

# Load the fine-tuned LoRA model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "qwen3_4b_thinking_meddies_lora",
    max_seq_length = 2048,
    load_in_4bit   = True,
)

# Enable fast inference mode
FastLanguageModel.for_inference(model)

In [ ]:
# Test questions — Vietnamese clinical scenarios
test_questions = [
    "Bệnh nhân bị đau bụng dữ dội vùng thượng vị kèm nôn, sốt 39 độ C. Cần làm gì để chẩn đoán và xử trí ban đầu?",
    "Giải thích cơ chế tác dụng của Metformin trong điều trị đái tháo đường type 2.",
    "Một bệnh nhân 65 tuổi bị tăng huyết áp kèm đái tháo đường, nên chọn nhóm thuốc hạ áp nào?",
]

for i, question in enumerate(test_questions):
    print(f"\n{'='*80}")
    print(f"Question {i+1}: {question}")
    print(f"{'='*80}")
    
    messages = [{"role": "user", "content": question}]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)
    
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=1024,
        temperature=0.6,
        top_p=0.95,
        do_sample=True,
    )
    
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    print(f"\nAnswer:\n{response}")

## 7. (Optional) Push to Hugging Face Hub

Uncomment and fill in your token/username to push the model.

In [ ]:
# from huggingface_hub import login
# login(token="YOUR_HF_TOKEN")

# # Push LoRA adapters
# model.push_to_hub("YOUR_USERNAME/qwen3-4b-thinking-meddies-lora", token="YOUR_HF_TOKEN")
# tokenizer.push_to_hub("YOUR_USERNAME/qwen3-4b-thinking-meddies-lora", token="YOUR_HF_TOKEN")

# # Push merged model
# model.push_to_hub_merged(
#     "YOUR_USERNAME/qwen3-4b-thinking-meddies-merged",
#     tokenizer,
#     save_method="merged_16bit",
#     token="YOUR_HF_TOKEN",
# )

# # Push GGUF
# model.push_to_hub_gguf(
#     "YOUR_USERNAME/qwen3-4b-thinking-meddies-gguf",
#     tokenizer,
#     quantization_method=["q4_k_m", "q8_0"],
#     token="YOUR_HF_TOKEN",
# )

## 8. Memory Stats

In [ ]:
import torch

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    max_mem = round(torch.cuda.max_memory_reserved(i) / 1024**3, 3)
    total_mem = round(props.total_memory / 1024**3, 3)
    print(f"GPU {i} ({props.name}): {max_mem} / {total_mem} GB reserved")